## Week 03 Homework Submission

**Mini-project goal:** 

* Build an agent that can search Wikipedia and fetch page content to answer questions

**Define tools**

* Create an agent that uses these tools to answer questions about any topic

## Wikipedia API setup
1. Search API - Find pages related to a topic
2. Fetch API - Fetch raw content of a page

In [ ]:
# Replace YOUR_QUERY with your search term. Use + for spaces (for example, lesser+capybara).
search_url = "https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srsearch=YOUR_QUERY"


# Replace PAGE_TITLE with the exact title of the Wikipedia page.
page_url = "https://en.wikipedia.org/w/index.php?title=PAGE_TITLE&action=raw"


In [2]:
import requests

# Wikipedia requires a User-Agent header on API requests. Without it, your requests may be blocked. Pass it in every requests.get call:
# headers = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}
# response = requests.get(search_url, headers=headers)


### Q1. Define the search tool

In [ ]:
def search_wikipedia(query: str) -> list[dict]:
    """
    Search Wikipedia for pages related to a query.

    Calls the Wikipedia search API and returns a list of search result objects,
    each containing fields like 'title', 'pageid', 'snippet', 'size', and 'timestamp'.

    Args:
        query: The search term to look up on Wikipedia.

    Returns:
        A list of dicts, one per search result, as returned by the Wikipedia API.
    """
    search_url = f"https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srsearch={query}"
    headers = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}
    response = requests.get(search_url, headers=headers)
    data = response.json()
    return data["query"]["search"]

In [34]:
query = "capybara"
search_results = search_wikipedia(query)

In [35]:
print(type(search_results))
print(f"total results for query {query}: {len(search_results)}")

<class 'list'>
total results for query capybara: 10


### Q2. Analyze Search Results

How many search results contain the word "capybara" (case-insensitive) in their title?

In [13]:
capybara_results = [search_results[i] for i in range(len(search_results)) if 'capybara' in search_results[i]['title'].lower()]

In [14]:
print(len(capybara_results))

5


### Q3. Define the Get Page Tool

Note: Wikipedia pages can be long. If you're using a smaller or free model with a limited context window, you may need to chunk the page content and only send the most relevant chunk to the LLM. The right chunk size depends on your model's input token limit.

In [ ]:
# Response object properties & methods: https://www.w3schools.com/python/ref_requests_response.asp

def get_page(page_title: str) -> str:
    """
    Fetch the raw wikitext content of a Wikipedia page.

    Calls the Wikipedia raw page API using the exact page title and returns
    the full page content as a wikitext string.

    Args:
        page_title: The exact title of the Wikipedia page (e.g. "Capybara").

    Returns:
        The raw wikitext content of the page as a string.
    """
    page_url = f"https://en.wikipedia.org/w/index.php?title={page_title}&action=raw"
    headers = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}
    response = requests.get(page_url, headers=headers)
    return response.text

In [30]:
page_content = get_page("Capybara")
print(type(page_content))
print(len(page_content))

<class 'str'>
36946


### Q4. Agent Setup

Create an agent with access to both tools:
1. search_wikipedia - search wikipedia pages
2. get_page - fetch the content of specific page

#### 4a. Choose an Agentic Framework
* ToyAIKit, PydanticAI, OpenAI Agents SDK, LangChain, etc.
* or implement the agentic loop yourself from scratch

#### 4b. Choose an LLM provider
* OpenAI, Anthropic, AWS Bedrock, etc.

In [ ]:
from pydantic_ai import Agent

In [ ]:
from typing import List, Dict
import requests

## Encapsulate tools in a class
class SearchTools:
    def __init__(self):
        pass

    def search_wikipedia(self, query: str) -> List[Dict]:
        """
        Search Wikipedia for pages related to a query.

        Calls the Wikipedia search API and returns a list of search result objects,
        each containing fields like 'title', 'pageid', 'snippet', 'size', and 'timestamp'.

        Args:
            query: The search term to look up on Wikipedia.

        Returns:
            A list of dicts, one per search result, as returned by the Wikipedia API.
        """
        search_url = f"https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srsearch={query}"
        headers = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}
        response = requests.get(search_url, headers=headers)
        data = response.json()
        return data["query"]["search"]

    def get_page(self, page_title: str) -> str:
        """
        Fetch the raw wikitext content of a Wikipedia page.

        Calls the Wikipedia raw page API using the exact page title and returns
        the full page content as a wikitext string.

        Args:
            page_title: The exact title of the Wikipedia page (e.g. "Capybara").

        Returns:
            The raw wikitext content of the page as a string.
        """
        page_url = f"https://en.wikipedia.org/w/index.php?title={page_title}&action=raw"
        headers = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}
        response = requests.get(page_url, headers=headers)
        return response.text

In [44]:
search_tools = SearchTools()

In [ ]:
instructions = """
You're a Wikipedia search assistant.

Answer the user query using Wikipedia pages found via your search tools.

If the user provides a Wikipedia link, fetch only that page and use only its content to answer — do not use any other search tools.

If you find relevant information, answer immediately — do not perform additional searches.

If you cannot find relevant information, refine your query and retry up to 2 more times (3 total attempts). If still unsuccessful, say "I have insufficient information to answer this question."

IMPORTANT: Answer using ONLY information explicitly stated in the fetched pages. Do not use your own training knowledge, do not infer, and do not fill gaps with assumptions. If the pages do not contain enough information to fully answer the question, say so explicitly rather than speculating.

At the end of your answer, list all Wikipedia pages you retrieved as a numbered list of URLs in the format: https://en.wikipedia.org/wiki/Page_Title
"""

tools = [search_tools.search_wikipedia, search_tools.get_page]

In [90]:
search_agent = Agent(
    name='search',
    model='openai:gpt-4o-mini',
    instructions=instructions,
    tools=tools
)


### Q5. Testing your Agent - Single Page

Use your agent to answer this question:
* "What is this page about? https://en.wikipedia.org/wiki/Capybara"

Your agent should:
* Use the get_page tool to fetch the Capybara page
* Provide a concise summary of what the page is about

What is the main topic of the Capybara page according to your agent?

In [ ]:
single_page_query = "What is this page about? https://en.wikipedia.org/wiki/Capybara"
q5_result = await search_agent.run(single_page_query)

In [ ]:
q5_messages = q5_result.all_messages()

## Inspect Message Structure
for m in q5_messages:
    print(m.kind)
    for p in m.parts:
        part_kind = p.part_kind
        if part_kind == 'user-prompt':
            print('USER:', p.content)
        if part_kind == 'tool-call':
            print('TOOL CALL:', p.tool_name, p.args)
        if part_kind == 'tool-return':
            print('TOOL RETURN:', p.tool_name)
        if part_kind == 'text':
            print(p.content)
    print()

print(q5_result.usage())

request
USER: What is this page about? https://en.wikipedia.org/wiki/Capybara

response
TOOL CALL: get_page {"page_title":"Capybara"}

request
TOOL RETURN: get_page

response
The Wikipedia page about capybaras provides detailed information on this animal, specifically the species known as *Hydrochoerus hydrochaeris*, which is recognized as the largest living rodent. Native to South America, except for Chile, capybaras are semiaquatic herbivores that thrive in various environments close to water bodies, such as savannas and rainy forest areas. The page covers aspects of their biology, social behavior, diet, reproduction, and conservation status.

Additionally, the article includes information about the etymology of the name "capybara," its ecological role, and its cultural significance in various regions, including their popularity in urban areas and as pets. The page also discusses their hunting for meat and skins, highlighting their role in local diets, particularly in regions like Ve

In [ ]:
print(q5_result.output)

The Wikipedia page about capybaras provides detailed information on this animal, specifically the species known as *Hydrochoerus hydrochaeris*, which is recognized as the largest living rodent. Native to South America, except for Chile, capybaras are semiaquatic herbivores that thrive in various environments close to water bodies, such as savannas and rainy forest areas. The page covers aspects of their biology, social behavior, diet, reproduction, and conservation status.

Additionally, the article includes information about the etymology of the name "capybara," its ecological role, and its cultural significance in various regions, including their popularity in urban areas and as pets. The page also discusses their hunting for meat and skins, highlighting their role in local diets, particularly in regions like Venezuela. For further specifics on their social structures, behaviors, and adaptations to habitats, the page contains extensive references and bibliographical citations.


### Q6. Testing Your Agent - Search Then Fetch

Test if your agent can use both tools together.
Ask your agent:
* "What are the main threats to capybara populations?"

Your agent should:
* First use search_wikipedia to find pages about capybara threats
* Then use get_page to fetch the relevant pages and extract specific threat information


How many total tool calls (search + fetch combined) did your agent make? Include the following details in your answer as well:
* Which tools it used and in what order
* The final answer your agent gave

 

In [77]:
search_fetch_query = "What are the main threats to capybara populations?"

In [ ]:
from pydantic_ai.messages import FunctionToolCallEvent

class NamedCallback:

    def __init__(self, agent):
        self.agent_name = agent.name

    async def print_function_calls(self, ctx, event):
        # Detect nested streams
        if hasattr(event, "__aiter__"):
            async for sub in event:
                await self.print_function_calls(ctx, sub)
            return

        if isinstance(event, FunctionToolCallEvent):
            tool_name = event.part.tool_name
            args = event.part.args
            print(f"TOOL CALL ({self.agent_name}): {tool_name}({args})")

    async def __call__(self, ctx, event):
        return await self.print_function_calls(ctx, event)

callback = NamedCallback(search_agent)

q6_result = await search_agent.run(
    search_fetch_query,
    event_stream_handler=callback
)

TOOL CALL (search): search_wikipedia({"query":"capybara threats"})
TOOL CALL (search): search_wikipedia({"query":"Capybara"})
TOOL CALL (search): get_page({"page_title":"Capybara"})


In [ ]:
q6_messages = q6_result.all_messages()

In [ ]:
## Inspect Message Structure
for m in q6_messages:
    print(m.kind)
    for p in m.parts:
        part_kind = p.part_kind
        if part_kind == 'user-prompt':
            print('USER:', p.content)
        if part_kind == 'tool-call':
            print('TOOL CALL:', p.tool_name, p.args)
        if part_kind == 'tool-return':
            print('TOOL RETURN:', p.tool_name)
        if part_kind == 'text':
            print(p.content)
    print()

print(q6_result.usage())

request
USER: What are the main threats to capybara populations?

response
TOOL CALL: search_wikipedia {"query":"capybara threats"}

request
TOOL RETURN: search_wikipedia

response
TOOL CALL: search_wikipedia {"query":"Capybara"}

request
TOOL RETURN: search_wikipedia

response
TOOL CALL: get_page {"page_title":"Capybara"}

request
TOOL RETURN: get_page

response
The main threats to capybara populations come from both natural and human-induced factors. According to the information retrieved:

1. **Hunting**: Capybaras are hunted for their meat and pelts. In some areas, this has led to reductions in their numbers.
   
2. **Competition with Livestock**: In agricultural areas, humans may kill capybaras because they perceive them as competition for grazing resources with livestock.

3. **Habitat Loss**: Although capybaras adapt well to urbanization and can be found in parks and zoos, the destruction of their natural wetlands for agriculture or urban development can pose a threat to their p

The main threats to capybara populations come from both natural and human-induced factors. According to the information retrieved:

1. **Hunting**: Capybaras are hunted for their meat and pelts. In some areas, this has led to reductions in their numbers.
   
2. **Competition with Livestock**: In agricultural areas, humans may kill capybaras because they perceive them as competition for grazing resources with livestock.

3. **Habitat Loss**: Although capybaras adapt well to urbanization and can be found in parks and zoos, the destruction of their natural wetlands for agriculture or urban development can pose a threat to their populations.

Despite these threats, capybara populations are generally considered stable across much of their South American range, and they have a high reproductive rate that supports population recovery. They also benefit from being farmed in some regions, which can help to ensure the protection of wetland habitats.
